In [20]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def make_student_success(n=600, seed=1):
    rng = np.random.default_rng(seed)

    X = pd.DataFrame({
        "hours": rng.integers(0, 12, n),
        "attendance": rng.integers(40, 100, n),
        "sleep": rng.integers(4, 9, n),
        "practice_tests": rng.integers(0, 6, n),
    })

    y = (((X["hours"] >= 5) &
          (X["attendance"] >= 70) &
          (X["sleep"] >= 6))
          |
          (X["practice_tests"] >= 4)).astype(int)

    y = np.where(rng.random(n) < 0.07, 1-y, y)
    return X, y

X, y = make_student_success()


#print(X.head())
#print(y[:5])



# ✅ Stratify keeps class balance similar in train/test (critical in classification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=3,stratify=y
)


model = LogisticRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

new_student = pd.DataFrame([{
    "hours": 4,
    "attendance": 50,
    "sleep": 7,
    "practice_tests":0
}])

print("Prediction:", int(model.predict(new_student)[0]))

print("Accuracy:", round(accuracy_score(y_test, pred), 3))



Prediction: 0
Accuracy: 0.833


In [9]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def make_student_success(n=600, seed=1):
    rng = np.random.default_rng(seed)
    X = pd.DataFrame({
        "hours": rng.integers(0, 12, n),
        "attendance": rng.integers(40, 100, n),
        "sleep": rng.integers(4, 9, n),
        "practice_tests": rng.integers(0, 6, n),
    })
    y = (((X["hours"] >= 5) &
          (X["attendance"] >= 70) &
          (X["sleep"] >= 6))
          |
          (X["practice_tests"] >= 4)).astype(int)
    # Add noise (real-life mistakes)
    y = np.where(rng.random(n) < 0.07, 1-y, y)
    return X, y

# Create dataset

X, y = make_student_success()


print(X.head())
print(y[:5])


seeds = [1, 2, 3, 4, 5, 10, 20, 30]
scores = []


# ✅ Stratify keeps class balance similar in train/test (critical in classification)

for s in seeds:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=s, stratify=y
    )

    model = LogisticRegression()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, pred)
    scores.append(acc)
    print("seed", s, "accuracy", round(acc, 3))

print("\nAvg accuracy:", round(float(np.mean(scores)), 3))
print("Min accuracy:", round(float(np.min(scores)), 3))
print("Max accuracy:", round(float(np.max(scores)), 3))



new_student = pd.DataFrame([{
    "hours": 4,
    "attendance": 50,
    "sleep": 7,
    "practice_tests":0
}])


print("Prediction:", int(model.predict(new_student)[0]))

#print("Accuracy:", round(accuracy_score(y_test, pred), 3))







   hours  attendance  sleep  practice_tests
0      5          94      6               5
1      6          56      5               0
2      9          66      8               1
3     11          73      4               3
4      0          69      7               4
[1 0 1 0 1]
seed 1 accuracy 0.825
seed 2 accuracy 0.775
seed 3 accuracy 0.833
seed 4 accuracy 0.808
seed 5 accuracy 0.783
seed 10 accuracy 0.867
seed 20 accuracy 0.825
seed 30 accuracy 0.783

Avg accuracy: 0.812
Min accuracy: 0.775
Max accuracy: 0.867
Prediction: 0


# Repeating Train/Test Splits with Different Seeds

## What’s Happening?

We are splitting the **same dataset multiple times** using different random seeds.

Each seed creates a **different train/test split**.

Example idea:

Seed 1 → different split  
Seed 2 → different split  
Seed 3 → different split  
Seed 4 → different split  

Even though the **dataset is identical**, the **rows selected for training and testing change**.

This means the model is tested under **multiple conditions**.

---

# Why Do This?

Because **one single split can lie to you**.

Imagine this scenario:

Seed 1 → lucky split → accuracy = 0.91  
Seed 2 → unlucky split → accuracy = 0.82  

Which one is correct?

**Answer: Neither.**

The **real performance** of the model is the **average across many splits**.

This is a **step toward robust evaluation**.

You are **stress-testing the model** instead of trusting one lucky experiment.

This is a **professional machine learning habit**.

---